# 03 — TF-IDF Representation

This notebook explains TF-IDF and demonstrates how it converts text to numerical features.

**Objectives:**
- Explain why text must be converted to numbers
- Describe TF, IDF, and their combination
- Demonstrate vocabulary construction
- Show the effect of different parameters
- Explain data leakage prevention via Pipeline

## 1. Why Convert Text to Numbers?

ML models require numerical inputs. TF-IDF converts text to weighted numerical vectors.

### Bag of Words (BoW)
Represents documents as word count vectors. Limitation: common words dominate.

### TF-IDF: An Improvement
Weights terms by importance:
- **TF**: How often a term appears in a document
- **IDF**: log(N / df(t)) — penalizes common terms
- **TF-IDF = TF x IDF**: Balances local importance with global rarity

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
print("Libraries loaded.")

## 2. Simple TF-IDF Example

In [ ]:
sample_docs = [
    "The president signed the new trade agreement today.",
    "Breaking news: president announces major policy change.",
    "Scientists discover new species in the Amazon rainforest.",
    "The stock market crashed after trade war escalation.",
]

vec = TfidfVectorizer()
X = vec.fit_transform(sample_docs)

print(f"Vocabulary size: {len(vec.vocabulary_)}")
print(f"Matrix shape: {X.shape} (documents x terms)")
print(f"Matrix type: {type(X).__name__} (sparse)")
print(f"Non-zero entries: {X.nnz} out of {X.shape[0] * X.shape[1]}")

In [ ]:
df_tfidf = pd.DataFrame(
    X.toarray(),
    columns=vec.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(sample_docs))]
)
print("TF-IDF Matrix:")
print(df_tfidf.round(3).to_string())

## 3. Unigrams vs. Bigrams

- **Unigrams**: single words ("fake", "news")
- **Bigrams**: word pairs ("fake news", "white house")

In [ ]:
vec_uni = TfidfVectorizer(ngram_range=(1, 1))
vec_bi = TfidfVectorizer(ngram_range=(1, 2))
X_uni = vec_uni.fit_transform(sample_docs)
X_bi = vec_bi.fit_transform(sample_docs)
print(f"Unigrams only: {X_uni.shape[1]} features")
print(f"Uni + Bigrams: {X_bi.shape[1]} features")
bigrams = [f for f in vec_bi.get_feature_names_out() if " " in f]
print(f"\nBigram features: {bigrams}")

## 4. Key Parameters

| Parameter | Purpose |
|-----------|--------|
| `lowercase=True` | Normalize case |
| `ngram_range=(1,2)` | Unigrams + bigrams |
| `max_features=50000` | Limit vocabulary |
| `min_df=3` | Ignore rare terms |
| `max_df=0.95` | Ignore very common terms |
| `sublinear_tf=True` | Log dampening |
| `stop_words='english'` | Remove stop words |

## 5. Sparse vs Dense Representations

TF-IDF produces **sparse matrices** where most values are zero. This is memory-efficient for text data.

## 6. TF-IDF vs Modern Embeddings

| Aspect | TF-IDF | Embeddings (BERT) |
|--------|--------|-------------------|
| Word order | No | Yes |
| Semantic similarity | No | Yes |
| Dimensionality | High | Low (768) |
| Computation | Fast | Slow |
| Interpretability | High | Low |

TF-IDF is well-suited here: interpretable, fast, works well with classic ML.

## 7. Data Leakage Prevention

TF-IDF is integrated in an sklearn Pipeline:

```python
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(...)),
    ('classifier', Model(...))
])
```

The Pipeline ensures:
1. `fit()` learns vocabulary from training data only
2. `transform()` uses that vocabulary on test data
3. Unknown test words are ignored
4. No information leaks from test to train

In [ ]:
from sklearn.linear_model import LogisticRegression

pipeline_demo = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=100)),
    ("classifier", LogisticRegression(max_iter=100))
])

print("Pipeline structure:")
for name, obj in pipeline_demo.named_steps.items():
    print(f"  {name}: {type(obj).__name__}")
print("\nThe Pipeline guarantees TF-IDF is fit only on training data.")

## 8. Project TF-IDF Configuration

```python
TFIDF_PARAMS = {
    'lowercase': True,
    'ngram_range': (1, 2),
    'max_features': 50000,
    'min_df': 3,
    'max_df': 0.95,
    'sublinear_tf': True,
    'stop_words': 'english',
}
```

These parameters balance expressiveness, noise reduction, and efficiency.